# Image Processing Techniques Implementation

This notebook demonstrates various image processing techniques including transformations, filtering, thresholding, edge detection, histogram equalization, and convolution operations.

The techniques are implemented from scratch where possible to deepen understanding of fundamental image processing concepts.

## 1. Loading Required Libraries and Images

First, let's import the necessary libraries and load an image for processing.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import io, color, filters, exposure
from scipy import ndimage

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)  # Larger default figure size
plt.style.use('default')

In [ ]:
# Function to display images side by side
def display_images(images, titles, cmaps=None, figsize=(15, 10)):
    n = len(images)
    if cmaps is None:
        cmaps = ['gray'] * n
        
    plt.figure(figsize=figsize)
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(images[i], cmap=cmaps[i])
        plt.title(titles[i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Load an image
# Replace with your own image path if needed
try:
    # Try loading a sample image
    image_path = "sample_image.jpg"  
    img = cv2.imread(image_path)
    
    # If image not found, download a sample image
    if img is None:
        raise FileNotFoundError
        
except FileNotFoundError:
    print("Using a sample Lena image instead...")
    from skimage import data
    img = data.astronaut()  # Using a built-in sample image

# Convert BGR to RGB if loaded with OpenCV
if len(img.shape) == 3 and img.shape[2] == 3 and isinstance(img, np.ndarray):
    if not isinstance(img, np.ndarray) or 'float' not in str(img.dtype):
        # This is likely a BGR image from OpenCV
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
# Create a grayscale version
img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img

# Display the original images
display_images([img], ['Original Image'], [None])
display_images([img_gray], ['Grayscale Image'], ['gray'])

## 2. Linear Transformations

Linear transformations are the simplest image processing operations that manipulate pixel values directly. We'll implement:
- Brightness adjustment (adding a constant)
- Contrast stretching (scaling pixel values)

In [ ]:
# Brightness adjustment
def adjust_brightness(image, value):
    """Adjust brightness by adding a constant value to all pixels."""
    # Create a copy to avoid modifying the original
    result = image.copy().astype(np.float32)
    
    # Add the value to all pixels
    result = result + value
    
    # Clip values to ensure they're within valid range [0, 255]
    result = np.clip(result, 0, 255).astype(np.uint8)
    
    return result

# Apply brightness adjustment
brightened = adjust_brightness(img_gray, 50)  # Increase brightness
darkened = adjust_brightness(img_gray, -50)   # Decrease brightness

# Display results
display_images([img_gray, brightened, darkened], 
               ['Original', 'Brightened (+50)', 'Darkened (-50)'], 
               ['gray', 'gray', 'gray'])

In [ ]:
# Contrast stretching
def stretch_contrast(image, a=0, b=255):
    """
    Apply contrast stretching to map the intensity values from range [min, max] 
    to a new range [a, b]
    """
    # Create a copy to avoid modifying the original
    result = image.copy().astype(np.float32)
    
    # Get the current min and max values
    min_val = np.min(result)
    max_val = np.max(result)
    
    # Apply the contrast stretching formula
    if max_val > min_val:
        result = (result - min_val) * ((b - a) / (max_val - min_val)) + a
    
    # Clip values and convert back to uint8
    return np.clip(result, 0, 255).astype(np.uint8)

# Apply contrast stretching
enhanced_contrast = stretch_contrast(img_gray)
reduced_contrast = stretch_contrast(img_gray, 50, 200)  # Reduced contrast range

# Display results
display_images([img_gray, enhanced_contrast, reduced_contrast], 
               ['Original', 'Full Contrast Stretch [0,255]', 'Limited Contrast [50,200]'], 
               ['gray', 'gray', 'gray'])

## 3. Basic Image Filtering

Filtering is a fundamental operation in image processing used for smoothing, noise reduction, and feature enhancement. We'll implement:
- Mean filtering (average of neighborhood pixels)
- Gaussian filtering (weighted average with Gaussian kernel)

In [ ]:
# Add some noise to better demonstrate the filtering effects
def add_noise(image, var=0.01):
    """Add Gaussian noise to the image"""
    row, col = image.shape
    mean = 0
    sigma = var**0.5
    gauss = np.random.normal(mean, sigma, (row, col))
    gauss = gauss.reshape(row, col)
    noisy = image + gauss * 255
    return np.clip(noisy, 0, 255).astype(np.uint8)

# Create noisy image
noisy_img = add_noise(img_gray, 0.02)

# Display the original and noisy images
display_images([img_gray, noisy_img], 
               ['Original Image', 'Noisy Image'], 
               ['gray', 'gray'])

In [ ]:
# Mean filtering implementation
def mean_filter(image, kernel_size=3):
    """Apply mean filter using a sliding window approach."""
    # Get image dimensions
    height, width = image.shape
    
    # Calculate padding size
    pad_size = kernel_size // 2
    
    # Create padded image
    padded_img = np.pad(image, ((pad_size, pad_size), (pad_size, pad_size)), mode='reflect')
    
    # Create output image
    filtered = np.zeros_like(image)
    
    # Apply filter
    for i in range(height):
        for j in range(width):
            # Extract neighborhood
            neighborhood = padded_img[i:i+kernel_size, j:j+kernel_size]
            
            # Calculate mean
            filtered[i, j] = np.mean(neighborhood)
            
    return filtered.astype(np.uint8)

# Apply mean filter with different kernel sizes
mean_filtered_3x3 = mean_filter(noisy_img, 3)
mean_filtered_5x5 = mean_filter(noisy_img, 5)

# Display results
display_images([noisy_img, mean_filtered_3x3, mean_filtered_5x5], 
               ['Noisy Image', 'Mean Filter (3×3)', 'Mean Filter (5×5)'], 
               ['gray', 'gray', 'gray'])

In [ ]:
# Gaussian filtering
def gaussian_filter(image, kernel_size=3, sigma=1.0):
    """Apply Gaussian filter using a sliding window approach."""
    # Create Gaussian kernel
    k = kernel_size // 2
    x, y = np.mgrid[-k:k+1, -k:k+1]
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()
    
    # Apply convolution
    filtered = ndimage.convolve(image, kernel)
    
    return filtered.astype(np.uint8)

# Apply Gaussian filter with different kernel sizes
gauss_filtered_3x3 = gaussian_filter(noisy_img, 3, 1.0)
gauss_filtered_5x5 = gaussian_filter(noisy_img, 5, 1.5)

# Display results
display_images([noisy_img, gauss_filtered_3x3, gauss_filtered_5x5], 
               ['Noisy Image', 'Gaussian Filter (3×3, σ=1.0)', 'Gaussian Filter (5×5, σ=1.5)'], 
               ['gray', 'gray', 'gray'])

In [ ]:
# Compare mean and Gaussian filters
display_images([noisy_img, mean_filtered_3x3, gauss_filtered_3x3], 
               ['Noisy Image', 'Mean Filter (3×3)', 'Gaussian Filter (3×3, σ=1.0)'], 
               ['gray', 'gray', 'gray'])

## 4. Global Thresholding

Thresholding is a segmentation technique used to separate objects from the background by converting grayscale images to binary images.

In [ ]:
# Global thresholding implementation
def global_threshold(image, threshold_value):
    """Apply global thresholding to create binary image."""
    # Create a copy of the image
    result = image.copy()
    
    # Apply thresholding
    result[result >= threshold_value] = 255
    result[result < threshold_value] = 0
    
    return result

# Apply global thresholding with different threshold values
thresh_low = global_threshold(img_gray, 50)
thresh_mid = global_threshold(img_gray, 127)
thresh_high = global_threshold(img_gray, 200)

# Display results
display_images([img_gray, thresh_low, thresh_mid, thresh_high], 
               ['Original', 'Threshold = 50', 'Threshold = 127', 'Threshold = 200'], 
               ['gray', 'gray', 'gray', 'gray'])

In [ ]:
# Otsu's method for automatic thresholding
def otsu_threshold(image):
    """Apply Otsu's method to find optimal threshold value."""
    # Calculate histogram
    hist, bins = np.histogram(image.flatten(), 256, [0, 256])
    
    # Total number of pixels
    total_pixels = image.size
    
    # Calculate sum of all intensities
    sum_total = sum(i * h for i, h in enumerate(hist))
    
    best_threshold = 0
    best_variance = 0
    sum_background = 0
    weight_background = 0
    
    # For each potential threshold value
    for threshold in range(256):
        # Calculate weights
        weight_background += hist[threshold]
        if weight_background == 0:
            continue
        
        weight_foreground = total_pixels - weight_background
        if weight_foreground == 0:
            break
        
        # Calculate means
        sum_background += threshold * hist[threshold]
        mean_background = sum_background / weight_background
        mean_foreground = (sum_total - sum_background) / weight_foreground
        
        # Calculate between-class variance
        variance = weight_background * weight_foreground * (mean_background - mean_foreground) ** 2
        
        # Update best threshold if current variance is higher
        if variance > best_variance:
            best_variance = variance
            best_threshold = threshold
    
    # Apply threshold
    result = global_threshold(image, best_threshold)
    
    return result, best_threshold

# Apply Otsu's thresholding
otsu_img, otsu_value = otsu_threshold(img_gray)

# Display results
display_images([img_gray, otsu_img], 
               [f'Original', f'Otsu\'s Threshold (value = {otsu_value})'], 
               ['gray', 'gray'])

## 5. Edge Detection

Edge detection identifies points in an image where the brightness changes sharply or has discontinuities. We'll implement:
- Sobel operators
- Prewitt operators

In [ ]:
# Sobel edge detection
def sobel_edge_detection(image):
    """Apply Sobel edge detection to detect gradients."""
    # Define Sobel kernels
    sobel_x = np.array([[-1, 0, 1], 
                         [-2, 0, 2], 
                         [-1, 0, 1]])
    
    sobel_y = np.array([[-1, -2, -1], 
                         [0, 0, 0], 
                         [1, 2, 1]])
    
    # Apply convolution for x and y directions
    grad_x = ndimage.convolve(image.astype(float), sobel_x)
    grad_y = ndimage.convolve(image.astype(float), sobel_y)
    
    # Calculate gradient magnitude
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)
    
    # Normalize to 0-255
    grad_magnitude = (grad_magnitude / grad_magnitude.max() * 255).astype(np.uint8)
    
    # Normalize directional gradients for visualization
    grad_x = np.abs(grad_x)
    grad_x = (grad_x / grad_x.max() * 255).astype(np.uint8)
    
    grad_y = np.abs(grad_y)
    grad_y = (grad_y / grad_y.max() * 255).astype(np.uint8)
    
    return grad_x, grad_y, grad_magnitude

# Apply Sobel edge detection
sobel_x, sobel_y, sobel_mag = sobel_edge_detection(img_gray)

# Display results
display_images([img_gray, sobel_x, sobel_y, sobel_mag], 
               ['Original Image', 'Sobel X', 'Sobel Y', 'Sobel Magnitude'], 
               ['gray', 'gray', 'gray', 'gray'])

In [ ]:
# Prewitt edge detection
def prewitt_edge_detection(image):
    """Apply Prewitt edge detection to detect gradients."""
    # Define Prewitt kernels
    prewitt_x = np.array([[-1, 0, 1], 
                           [-1, 0, 1], 
                           [-1, 0, 1]])
    
    prewitt_y = np.array([[-1, -1, -1], 
                           [0, 0, 0], 
                           [1, 1, 1]])
    
    # Apply convolution for x and y directions
    grad_x = ndimage.convolve(image.astype(float), prewitt_x)
    grad_y = ndimage.convolve(image.astype(float), prewitt_y)
    
    # Calculate gradient magnitude
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)
    
    # Normalize to 0-255
    grad_magnitude = (grad_magnitude / grad_magnitude.max() * 255).astype(np.uint8)
    
    # Normalize directional gradients for visualization
    grad_x = np.abs(grad_x)
    grad_x = (grad_x / grad_x.max() * 255).astype(np.uint8)
    
    grad_y = np.abs(grad_y)
    grad_y = (grad_y / grad_y.max() * 255).astype(np.uint8)
    
    return grad_x, grad_y, grad_magnitude

# Apply Prewitt edge detection
prewitt_x, prewitt_y, prewitt_mag = prewitt_edge_detection(img_gray)

# Display results
display_images([img_gray, prewitt_x, prewitt_y, prewitt_mag], 
               ['Original Image', 'Prewitt X', 'Prewitt Y', 'Prewitt Magnitude'], 
               ['gray', 'gray', 'gray', 'gray'])

In [ ]:
# Compare Sobel and Prewitt edge detection
display_images([img_gray, sobel_mag, prewitt_mag], 
               ['Original', 'Sobel Edges', 'Prewitt Edges'], 
               ['gray', 'gray', 'gray'])

## 6. Histogram Equalization

Histogram equalization is a technique for adjusting image contrast by redistributing intensity values.

In [ ]:
# Histogram calculation and visualization
def plot_histogram(image, title="Histogram", bins=256):
    """Calculate and display histogram of an image."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Display image
    ax1.imshow(image, cmap='gray')
    ax1.set_title('Image')
    ax1.axis('off')
    
    # Calculate and display histogram
    hist, bins = np.histogram(image.flatten(), bins, [0, bins])
    ax2.bar(range(len(hist)), hist, width=1.0)
    ax2.set_xlim([0, bins])
    ax2.set_title(title)
    ax2.set_xlabel('Pixel Value')
    ax2.set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    return hist

# Display histogram of the original image
hist_original = plot_histogram(img_gray, "Original Histogram")

In [ ]:
# Histogram equalization implementation
def histogram_equalization(image):
    """Apply histogram equalization to enhance contrast."""
    # Calculate histogram
    hist, bins = np.histogram(image.flatten(), 256, [0, 256])
    
    # Calculate cumulative distribution function (CDF)
    cdf = hist.cumsum()
    
    # Normalize CDF to range [0, 255]
    cdf_normalized = (cdf * 255 / cdf[-1]).astype(np.uint8)
    
    # Map pixel values using the CDF
    equalized_img = cdf_normalized[image]
    
    return equalized_img

# Apply histogram equalization
equalized_img = histogram_equalization(img_gray)

# Display results
display_images([img_gray, equalized_img], 
               ['Original Image', 'Histogram Equalized'], 
               ['gray', 'gray'])

# Display histogram of the equalized image
hist_equalized = plot_histogram(equalized_img, "Equalized Histogram")

## 7. 2D Convolution

Convolution is the basis for many image processing operations. We'll implement a custom 2D convolution function and apply different kernels.

In [ ]:
# Custom 2D convolution implementation
def convolution_2d(image, kernel):
    """Apply custom 2D convolution with the given kernel."""
    # Get image and kernel dimensions
    i_height, i_width = image.shape
    k_height, k_width = kernel.shape
    
    # Padding size
    pad_h = k_height // 2
    pad_w = k_width // 2
    
    # Create padded image
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    
    # Create output image
    output = np.zeros_like(image, dtype=float)
    
    # Apply convolution
    for i in range(i_height):
        for j in range(i_width):
            # Extract region of interest
            roi = padded[i:i+k_height, j:j+k_width]
            
            # Apply kernel
            output[i, j] = np.sum(roi * kernel)
    
    # Normalize if needed
    if np.sum(kernel) != 0:
        output = output / np.sum(kernel)
    
    # Clip values to [0, 255] and convert to uint8
    return np.clip(output, 0, 255).astype(np.uint8)

In [ ]:
# Apply sharpening kernel
sharpening_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

# Apply the sharpening kernel
sharpened_img = convolution_2d(img_gray, sharpening_kernel)

# Display results
display_images([img_gray, sharpened_img], 
               ['Original Image', 'Sharpened Image'], 
               ['gray', 'gray'])

In [ ]:
# Define and apply other kernels
# Box blur kernel
box_blur = np.ones((3, 3)) / 9

# Gaussian blur kernel (3x3)
gaussian_blur = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
]) / 16

# Edge detection kernel
edge_kernel = np.array([
    [-1, -1, -1],
    [-1, 8, -1],
    [-1, -1, -1]
])

# Emboss kernel
emboss_kernel = np.array([
    [-2, -1, 0],
    [-1, 1, 1],
    [0, 1, 2]
])

# Apply different kernels
box_blurred = convolution_2d(img_gray, box_blur)
gaussian_blurred = convolution_2d(img_gray, gaussian_blur)
edge_detected = convolution_2d(img_gray, edge_kernel)
embossed = convolution_2d(img_gray, emboss_kernel)

# Display results
display_images([img_gray, box_blurred, gaussian_blurred, edge_detected, embossed], 
               ['Original', 'Box Blur', 'Gaussian Blur', 'Edge Detection', 'Emboss'], 
               ['gray', 'gray', 'gray', 'gray', 'gray'],
               figsize=(18, 10))

## 8. Comparing All Techniques

Let's display all the processed images together for comparison.

In [ ]:
# Create a comprehensive comparison grid
all_images = [
    img_gray,               # Original
    brightened,             # Brightened
    enhanced_contrast,      # Contrast stretched
    gauss_filtered_3x3,     # Gaussian filtered
    otsu_img,               # Thresholded (Otsu)
    sobel_mag,              # Edge detected (Sobel)
    equalized_img,          # Histogram equalized
    sharpened_img           # Sharpened (Convolution)
]

all_titles = [
    'Original',
    'Brightened',
    'Contrast Stretched',
    'Gaussian Filtered',
    'Otsu Thresholding',
    'Edge Detection (Sobel)',
    'Histogram Equalized',
    'Sharpened'
]

plt.figure(figsize=(20, 15))
rows = 2
cols = 4

for i, (image, title) in enumerate(zip(all_images, all_titles)):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.subplots_adjust(wspace=0.1, hspace=0.2)
plt.show()

## Conclusion

In this notebook, we've implemented and explored various image processing techniques:

1. **Linear Transformations** - Brightness adjustment and contrast stretching
2. **Basic Image Filtering** - Mean and Gaussian filtering for noise reduction
3. **Global Thresholding** - Binary segmentation using manual and Otsu's thresholding
4. **Edge Detection** - Using Sobel and Prewitt operators
5. **Histogram Equalization** - Enhancing image contrast
6. **2D Convolution** - Custom implementation with various kernels

These techniques form the foundation of more advanced image processing and computer vision applications.